<a href="https://colab.research.google.com/github/edwinfu9999/watermelonwhisperer/blob/main/watermelon_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model

# =========================
# Mount Drive (Your Provided Block)
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# Collect file paths ONLY (fast)
# =========================

dataset_dir = "/content/drive/MyDrive/19_datasets"

wav_paths = []
jpg_paths = []
labels = []

subdirs = [d for d in os.listdir(dataset_dir)
           if os.path.isdir(os.path.join(dataset_dir, d))]

for subdir in subdirs:
    data_id, label = subdir.split("_")
    label = float(label)

    chu_dir = os.path.join(dataset_dir, subdir, "chu")
    folders = [f for f in os.listdir(chu_dir)
               if os.path.isdir(os.path.join(chu_dir, f))]

    for folder in folders:
        folder_path = os.path.join(chu_dir, folder)

        wav_file = [f for f in os.listdir(folder_path) if f.endswith(".wav")][0]
        jpg_file = [f for f in os.listdir(folder_path) if f.endswith(".jpg")][0]

        wav_paths.append(os.path.join(folder_path, wav_file))
        jpg_paths.append(os.path.join(folder_path, jpg_file))
        labels.append(label)

print(f"✅ Successfully cached paths for {len(labels)} samples.")

Mounted at /content/drive
✅ Successfully cached paths for 1557 samples.


In [ ]:
import tensorflow as tf

# Hyperparameters for preprocessing
IMG_SIZE = (224, 224)
TARGET_SAMPLE_RATE = 16000
AUDIO_DURATION_SAMP = TARGET_SAMPLE_RATE * 2

# ==========================================
# 1. BASE LOADING (Runs ONCE per file and gets Cached)
# ==========================================
def load_base_elements(jpg_path, wav_path, label):
    """Reads and decodes files. This heavy I/O part will be cached."""
    # Process Image
    img = tf.io.read_file(jpg_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)

    # Process Audio (Decode & Pad only)
    audio_binary = tf.io.read_file(wav_path)
    waveform, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)
    waveform = tf.squeeze(waveform, axis=-1)

    waveform = waveform[:AUDIO_DURATION_SAMP]
    zero_padding = tf.zeros([AUDIO_DURATION_SAMP] - tf.shape(waveform), dtype=tf.float32)
    waveform = tf.concat([waveform, zero_padding], axis=0)

    return img, waveform, label

# ==========================================
# 2. DYNAMIC AUGMENTATION (Runs EVERY Epoch)
# ==========================================
def add_white_noise(waveform):
    """Dynamically injects white noise using tf.cond based on a random SNR."""
    def apply_noise():
        signal_power = tf.reduce_mean(tf.square(waveform))
        # Pick a random SNR between 10dB (Noisy) and 25dB (Clean)
        snr_db = tf.random.uniform((), minval=10.0, maxval=25.0)
        snr_linear = tf.math.pow(10.0, snr_db / 10.0)

        noise_power = signal_power / (snr_linear + 1e-8)
        noise = tf.random.normal(
            shape=tf.shape(waveform),
            mean=0.0,
            stddev=tf.math.sqrt(noise_power)
        )
        return waveform + noise

    # 50% chance to augment the waveform
    return tf.cond(tf.random.uniform(()) > 0.5, apply_noise, lambda: waveform)

def finalize_train(img, waveform, label):
    """Runs AFTER cache: adds dynamic noise and computes STFT for training."""
    waveform = add_white_noise(waveform)

    # Compute STFT on the (potentially noisy) waveform
    stft = tf.signal.stft(waveform, frame_length=256, frame_step=128)
    spectrogram = tf.abs(stft)

    return {"image_input": img, "audio_input": spectrogram}, label

def finalize_val(img, waveform, label):
    """Runs AFTER cache: computes clean STFT for validation (NO NOISE)."""
    stft = tf.signal.stft(waveform, frame_length=256, frame_step=128)
    spectrogram = tf.abs(stft)

    return {"image_input": img, "audio_input": spectrogram}, label

# ==========================================
# 3. PIPELINE CONSTRUCTION
# ==========================================
full_dataset = tf.data.Dataset.from_tensor_slices((jpg_paths, wav_paths, labels))
full_dataset = full_dataset.shuffle(buffer_size=len(labels), seed=42)

val_size = int(len(labels) * 0.2)
train_dataset = full_dataset.skip(val_size)
val_dataset = full_dataset.take(val_size)

BATCH_SIZE = 16

# Train Pipeline
train_ds = (train_dataset
            .map(load_base_elements, num_parallel_calls=tf.data.AUTOTUNE)
            .cache() # <--- Caches the raw Google Drive reads in RAM
            .map(finalize_train, num_parallel_calls=tf.data.AUTOTUNE) # <--- Dynamically augments on the fly
            .batch(BATCH_SIZE)
            .prefetch(buffer_size=tf.data.AUTOTUNE))

# Validation Pipeline
val_ds = (val_dataset
          .map(load_base_elements, num_parallel_calls=tf.data.AUTOTUNE)
          .cache()
          .map(finalize_val, num_parallel_calls=tf.data.AUTOTUNE) # <--- Validates strictly on clean audio
          .batch(BATCH_SIZE)
          .prefetch(buffer_size=tf.data.AUTOTUNE))

In [ ]:
# L2 weight decay strength
l2_decay = regularizers.l2(1e-4)

# ==========================================
# 1. IMAGE BRANCH (MobileNetV2 + Augmentation)
# ==========================================
image_input = layers.Input(shape=(224, 224, 3), name="image_input")

# Data Augmentation (overfitting prevention, runs only during training)
augmented = layers.RandomFlip("horizontal")(image_input)
augmented = layers.RandomRotation(0.15)(augmented)

# Lightweight CNN feature extractor for low-latency
base_cnn = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_cnn.trainable = False  # Freeze pre-trained weights to prevent overfitting

cnn_features = tf.keras.applications.mobilenet_v2.preprocess_input(augmented)
cnn_features = base_cnn(cnn_features, training=False)
cnn_features = layers.GlobalAveragePooling2D()(cnn_features)
cnn_features = layers.Dense(128, activation="relu", kernel_regularizer=l2_decay)(cnn_features)
cnn_features = layers.BatchNormalization()(cnn_features)
cnn_features = layers.Dropout(0.4)(cnn_features)

# ==========================================
# 2. AUDIO BRANCH (Bidirectional LSTM)
# ==========================================
# Input dimensions are determined dynamically from the STFT shape (e.g., (249, 129))
audio_input = layers.Input(shape=(None, 129), name="audio_input")

# Bidirectional LSTMs capture past-and-future context of the audio thump signals
lstm_features = layers.Bidirectional(
    layers.LSTM(64, return_sequences=False, kernel_regularizer=l2_decay)
)(audio_input)
lstm_features = layers.BatchNormalization()(lstm_features)
lstm_features = layers.Dropout(0.4)(lstm_features)

# ==========================================
# 3. CONCATENATION & REGRESSION HEAD
# ==========================================
fused_features = layers.concatenate([cnn_features, lstm_features])

dense_head = layers.Dense(64, activation="relu", kernel_regularizer=l2_decay)(fused_features)
dense_head = layers.BatchNormalization()(dense_head)
dense_head = layers.Dropout(0.3)(dense_head)

# Linear output activation layer for continuous regression (sweetness scale index)
output_layer = layers.Dense(1, activation="linear", name="sweetness_output")(dense_head)

# Compile Model
model = Model(inputs=[image_input, audio_input], outputs=output_layer, name="SweetnessMultimodalFusion")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "SweetnessMultimodalFusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip         │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (RandomFlip)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation     │ (None, 224, 224,  │          0 │ random_flip[0][0] │
│ (RandomRotation)    │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide         │ (None, 224, 224,  │          0 │ random_rotation[… │
│ (TrueDivide)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract (Subtract) │ (None, 224, 224,  │          0 │ true_divide[0][0] │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ subtract[0][0]    │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_input         │ (None, None, 129) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    163,968 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     99,328 │ audio_input[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ bidirectional[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     16,448 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,539,073 (9.69 MB)

 Trainable params: 280,449 (1.07 MB)

 Non-trainable params: 2,258,624 (8.62 MB)

In [ ]:
callbacks = [
    # Halts training when validation loss stops dropping
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    # Drop learning rate to pass tricky flat surfaces on loss landscapes
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# Run training (uses our highly-optimized tf.data caches)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1141s 14s/step - loss: 100.9976 - mae: 9.9047 - val_loss: 88.2435 - val_mae: 9.3474 - learning_rate: 0.0010
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - loss: 82.2300 - mae: 8.9149 - val_loss: 63.3807 - val_mae: 7.9076 - learning_rate: 0.0010
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - loss: 58.2393 - mae: 7.4019 - val_loss: 37.9013 - val_mae: 6.0926 - learning_rate: 0.0010
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 77ms/step - loss: 31.5336 - mae: 5.2074 - val_loss: 17.6381 - val_mae: 4.0891 - learning_rate: 0.0010
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 5s 66ms/step - loss: 13.8931 - mae: 3.1545 - val_loss: 6.2916 - val_mae: 2.3760 - learning_rate: 0.0010
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 81ms/step - loss: 7.3219 - mae: 2.1064 - val_loss: 2.1178 - val_mae: 1.2241 - learning_rate: 0.0010
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - loss: 5.8638 - mae: 1.8662 - val_loss: 1.3282 - val_mae: 0.9611 - learning_rate: 0.0010
E

In [ ]:
# 1. Define the save path on your Google Drive
keras_save_path = os.path.join(dataset_dir, "watermelon_model(WhiteNoise).keras")

# 2. Save the model
print(f"💾 Saving model to Google Drive at: {keras_save_path}...")
model.save(keras_save_path)
print("✅ Model successfully saved to your Google Drive!")

💾 Saving model to Google Drive at: /content/drive/MyDrive/19_datasets/watermelon_model(WhiteNoise).keras...
✅ Model successfully saved to your Google Drive!


In [ ]:
import os
import tensorflow as tf
from google.colab import drive

# ==========================================
# 0. RE-MOUNT DRIVE (Mandatory after runtime reset)
# ==========================================
print("🔄 Reset detected. Re-mounting Google Drive to access persistent storage...")
drive.mount('/content/drive')

# ==========================================
# 1. DEFINE PATHS
# ==========================================
# The actual folder where you want to save the final asset
output_dir = "/content/drive/MyDrive"

# The path to your saved Keras model file
model_keras_path = "/content/drive/MyDrive/19_datasets/watermelon_model(WhiteNoise).keras"

# Clean output path: /content/drive/MyDrive/watermelon_sweetness_optimized.tflite
tflite_output_path = os.path.join(output_dir, "watermelon_sweetness_optimized(whiteNoise).tflite")

# ==========================================
# 2. VERIFY & LOAD FROM DRIVE
# ==========================================
if not os.path.exists(model_keras_path):
    raise FileNotFoundError(
        f"Could not locate '{model_keras_path}' on your Google Drive. "
        "Ensure it finished saving completely before you switched runtimes."
    )

print("🔄 Loading model from Google Drive (CPU environment mode)...")
# Loads the model file straight from the Drive disk into the fresh CPU RAM layout
model = tf.keras.models.load_model(model_keras_path)

# ==========================================
# 3. TFLITE CONVERSION & QUANTIZATION
# ==========================================
print("📦 Preparing TFLite Converter...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Apply latency reduction optimizations (Quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Handle target specs for LSTM / Audio components
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

print("⚡ Converting model to quantized TFLite format (running on CPU)...")
tflite_quantized_model = converter.convert()

# ==========================================
# 4. EXPORT BACK TO DRIVE
# ==========================================
print(f"💾 Writing optimized model back to Drive...")
with open(tflite_output_path, "wb") as f:
    f.write(tflite_quantized_model)

print(f"🎉 Success! Your deployment-ready model is safely stored at:\n👉 {tflite_output_path}")

🔄 Reset detected. Re-mounting Google Drive to access persistent storage...
Mounted at /content/drive
🔄 Loading model from Google Drive (CPU environment mode)...
📦 Preparing TFLite Converter...
⚡ Converting model to quantized TFLite format (running on CPU)...
Saved artifact at '/tmp/tmpu4klf8n2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image_input'), TensorSpec(shape=(None, None, 129), dtype=tf.float32, name='audio_input')]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137928106947792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137928106949712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137928106947216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137928106949904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137928106949520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137928106948368: TensorSp